# 00b. 확률과 통계 기초 (Probability & Statistics Foundations)

이번 단원에서는 머신러닝에 필요한 **확률과 통계 기초**를 배워보겠습니다.

## 학습 목표
- 확률 분포의 개념과 종류
- 조건부 확률과 베이즈 정리
- 최대우도 추정 (MLE)
- 손실 함수와 확률 분포의 연결
- 정규화와 통계의 관계

## 왜 이 단원이 필요한가?

머신러닝은 본질적으로 **확률론적 모델링**입니다. 모델은 데이터의 확률 분포를 학습하고, 예측은 확률로 표현됩니다.

- **CrossEntropy Loss**: 왜 분류 문제에서 사용하는가? → 다항 분포와 MLE
- **MSE Loss**: 왜 회귀 문제에서 사용하는가? → 정규 분포와 MLE
- **정규화**: 왜 BatchNorm이 작동하는가? → 통계적 정규화
- **불확실성**: 모델의 예측이 얼마나 확실한가? → 확률 해석

이 단원을 이해하면 손실 함수가 단순한 수식이 아니라 **확률론적 의미**를 가진다는 것을 알게 됩니다.


## 이 단원을 배우기 전에

**이전 단원 (00a) 복습**: 미분, Gradient Descent의 개념을 이해하셨나요?

## 이 단원 다음에는

**다음 단원 (01)**: 수학 기초를 다졌으니, 이제 PyTorch의 텐서를 배워봅시다.


---
# 1. 직관적 이해 (Why)
---

## 1.1 확률이란 무엇인가?

### 실생활 비유: 날씨 예보

- **확실성**: "내일 해가 뜬다" (확률 = 1)
- **불확실성**: "내일 비가 올 확률은 70%" (확률 = 0.7)
- **확률**: 불확실한 사건이 일어날 가능성을 0~1 사이의 숫자로 표현

### 머신러닝에서의 확률

- **분류 모델**: "이 이미지가 고양이일 확률은 0.85"
- **예측의 불확실성**: 확률이 높을수록 모델이 확신함
- **데이터의 패턴**: 데이터가 따르는 확률 분포를 학습

### 왜 딥러닝에 필요한가?

1. **손실 함수의 의미**: CrossEntropy는 확률 분포 간의 차이
2. **모델의 출력**: Softmax는 확률 분포로 변환
3. **학습의 목표**: 실제 데이터 분포를 모델이 근사


---
# 2. 수학적 기초 (What)
---

## 2.1 확률의 기본 개념 (고등학교 복습)

### 확률의 정의

사건 A가 일어날 확률:

$$P(A) = \frac{\text{A가 일어나는 경우의 수}}{\text{전체 경우의 수}}$$

### 확률의 성질

1. $0 \leq P(A) \leq 1$ (확률은 0과 1 사이)
2. $P(\text{전체}) = 1$ (전체 확률의 합은 1)
3. $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ (합사건)

### 조건부 확률

사건 B가 일어났을 때 A가 일어날 확률:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

**의미**: B가 주어졌을 때 A의 확률

### 베이즈 정리

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

**의미**: 
- $P(A)$: 사전 확률 (prior)
- $P(A|B)$: 사후 확률 (posterior)
- $P(B|A)$: 우도 (likelihood)


## 2.2 확률 분포

### 이산 확률 분포

**확률 질량 함수 (PMF)**: $P(X = x)$

#### 1) 이항 분포 (Binomial Distribution)

$n$번 시행에서 성공 횟수의 분포:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**예시**: 동전을 10번 던져 앞면이 나오는 횟수

#### 2) 다항 분포 (Multinomial Distribution)

여러 범주 중 하나를 선택하는 분포:

$$P(X_1=n_1, \ldots, X_k=n_k) = \frac{n!}{n_1! \cdots n_k!} p_1^{n_1} \cdots p_k^{n_k}$$

**머신러닝 연결**: 분류 문제의 정답 레이블 (one-hot encoding)

### 연속 확률 분포

**확률 밀도 함수 (PDF)**: $f(x)$, $P(a \leq X \leq b) = \int_a^b f(x) dx$

#### 1) 정규 분포 (Normal Distribution)

$$f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- $\mu$: 평균 (mean)
- $\sigma^2$: 분산 (variance)
- 표기: $X \sim \mathcal{N}(\mu, \sigma^2)$

**특징**:
- 종 모양 (bell curve)
- 평균 주변에 데이터가 집중
- 중심극한정리: 많은 확률변수의 합은 정규분포에 근사

**머신러닝 연결**: 
- 가중치 초기화 (Xavier, He initialization)
- 회귀 문제의 오차 분포
- BatchNorm의 정규화


## 2.3 최대우도 추정 (Maximum Likelihood Estimation, MLE)

### 우도 (Likelihood)

주어진 데이터가 특정 모델에서 나올 확률:

$$L(\theta | \mathbf{x}) = P(\mathbf{x} | \theta)$$

- $\theta$: 모델의 파라미터
- $\mathbf{x}$: 관측된 데이터

### MLE의 아이디어

**데이터를 가장 잘 설명하는 파라미터를 찾자!**

$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta L(\theta | \mathbf{x})$$

### 로그 우도

계산 편의를 위해 로그를 취함:

$$\log L(\theta | \mathbf{x}) = \sum_{i=1}^n \log P(x_i | \theta)$$

**장점**: 곱셈이 덧셈으로 변환, 수치적 안정성

### MLE와 손실 함수의 연결

**핵심**: 손실 함수를 최소화 = 로그 우도를 최대화

$$\text{Loss} = -\log L(\theta | \mathbf{x})$$

이것이 **Negative Log-Likelihood (NLL)**입니다!


## 2.4 손실 함수와 확률 분포의 연결

### 1) MSE Loss ↔ 정규 분포

**회귀 문제**: $y = f(x) + \epsilon$, $\epsilon \sim \mathcal{N}(0, \sigma^2)$

우도:

$$P(y | x, \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y - f(x))^2}{2\sigma^2}\right)$$

로그 우도:

$$\log P(y | x, \theta) = -\frac{(y - f(x))^2}{2\sigma^2} + \text{const}$$

**최대화 = MSE 최소화**:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2$$

### 2) CrossEntropy Loss ↔ 다항 분포

**분류 문제**: 클래스 $k$ 중 하나 선택

모델 출력 (Softmax):

$$\hat{p}_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}$$

실제 레이블 (one-hot): $y = [0, \ldots, 1, \ldots, 0]$

로그 우도:

$$\log P(y | x, \theta) = \sum_{k=1}^K y_k \log \hat{p}_k$$

**최대화 = CrossEntropy 최소화**:

$$\text{CrossEntropy} = -\sum_{k=1}^K y_k \log \hat{p}_k$$

### 핵심 통찰

**손실 함수는 확률 분포의 가정에서 자연스럽게 유도됩니다!**

- MSE: 오차가 정규분포를 따른다고 가정
- CrossEntropy: 레이블이 다항분포를 따른다고 가정


---
# 3. PyTorch 구현 (How)
---

## 3.1 확률 분포 시각화


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# 정규 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) 다양한 평균
x = np.linspace(-10, 10, 200)
for mu in [-2, 0, 2]:
    y = stats.norm.pdf(x, mu, 1)
    axes[0].plot(x, y, label=f'μ={mu}, σ²=1', linewidth=2)
axes[0].set_title('평균이 다른 정규분포')
axes[0].set_xlabel('x')
axes[0].set_ylabel('확률 밀도')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2) 다양한 분산
for sigma in [0.5, 1, 2]:
    y = stats.norm.pdf(x, 0, sigma)
    axes[1].plot(x, y, label=f'μ=0, σ²={sigma**2}', linewidth=2)
axes[1].set_title('분산이 다른 정규분포')
axes[1].set_xlabel('x')
axes[1].set_ylabel('확률 밀도')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3) 표준 정규분포와 68-95-99.7 규칙
x = np.linspace(-4, 4, 200)
y = stats.norm.pdf(x, 0, 1)
axes[2].plot(x, y, 'b-', linewidth=2, label='표준정규분포')
axes[2].fill_between(x, 0, y, where=(x >= -1) & (x <= 1), alpha=0.3, label='68% (±1σ)')
axes[2].fill_between(x, 0, y, where=(x >= -2) & (x <= 2), alpha=0.2, label='95% (±2σ)')
axes[2].set_title('68-95-99.7 규칙')
axes[2].set_xlabel('x')
axes[2].set_ylabel('확률 밀도')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("정규분포의 특징:")
print("- 평균(μ): 분포의 중심")
print("- 분산(σ²): 퍼진 정도")
print("- 68%의 데이터가 ±1σ 안에")
print("- 95%의 데이터가 ±2σ 안에")


## 3.2 MSE와 정규분포의 연결


In [ ]:
# MSE Loss와 정규분포
import torch
import torch.nn as nn

# 간단한 회귀 문제
torch.manual_seed(42)
x = torch.randn(100, 1)
y_true = 2 * x + 1 + 0.5 * torch.randn(100, 1)  # y = 2x + 1 + noise

# 모델 예측
model = nn.Linear(1, 1)
y_pred = model(x)

# MSE Loss 계산
mse_loss = nn.MSELoss()
loss = mse_loss(y_pred, y_true)

print("MSE Loss와 정규분포의 연결:")
print(f"MSE Loss: {loss.item():.4f}")
print("\n해석:")
print("- MSE를 최소화 = 오차의 제곱합을 최소화")
print("- 오차가 정규분포 N(0, σ²)를 따른다고 가정")
print("- MLE 관점: 데이터의 로그 우도를 최대화")

# 오차 분포 시각화
with torch.no_grad():
    errors = (y_pred - y_true).numpy().flatten()

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(x.numpy(), y_true.numpy(), alpha=0.5, label='실제 데이터')
plt.scatter(x.numpy(), y_pred.detach().numpy(), alpha=0.5, label='모델 예측')
plt.xlabel('x')
plt.ylabel('y')
plt.title('회귀 문제')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(errors, bins=20, density=True, alpha=0.7, label='실제 오차')
x_range = np.linspace(errors.min(), errors.max(), 100)
plt.plot(x_range, stats.norm.pdf(x_range, errors.mean(), errors.std()), 
         'r-', linewidth=2, label='정규분포 근사')
plt.xlabel('오차')
plt.ylabel('밀도')
plt.title('오차 분포 (정규분포에 근사)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3.3 CrossEntropy와 다항분포의 연결


In [ ]:
# CrossEntropy Loss와 다항분포
import torch
import torch.nn.functional as F

# 3-클래스 분류 문제
logits = torch.tensor([[2.0, 1.0, 0.1]])  # 모델의 원시 출력
true_label = torch.tensor([0])  # 실제 레이블 (클래스 0)

# Softmax: 확률 분포로 변환
probs = F.softmax(logits, dim=1)
print("Softmax 출력 (확률 분포):")
print(f"  P(클래스 0) = {probs[0, 0].item():.4f}")
print(f"  P(클래스 1) = {probs[0, 1].item():.4f}")
print(f"  P(클래스 2) = {probs[0, 2].item():.4f}")
print(f"  합계: {probs.sum().item():.4f}")

# CrossEntropy Loss
ce_loss = F.cross_entropy(logits, true_label)
print(f"\nCrossEntropy Loss: {ce_loss.item():.4f}")

# 수동 계산
manual_loss = -torch.log(probs[0, true_label])
print(f"수동 계산: -log(P(클래스 {true_label.item()})) = {manual_loss.item():.4f}")

print("\n해석:")
print("- Softmax: 로짓을 확률 분포로 변환 (합=1)")
print("- CrossEntropy: 정답 클래스의 확률이 높을수록 손실 감소")
print("- MLE 관점: 정답 클래스의 로그 우도를 최대화")

# 확률에 따른 손실 변화
probs_range = np.linspace(0.01, 1, 100)
losses = -np.log(probs_range)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.bar(['클래스 0', '클래스 1', '클래스 2'], probs[0].detach().numpy())
plt.ylabel('확률')
plt.title('Softmax 출력')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
plt.plot(probs_range, losses, linewidth=2)
plt.axvline(x=probs[0, 0].item(), color='r', linestyle='--', 
            label=f'현재 확률={probs[0, 0].item():.3f}')
plt.xlabel('정답 클래스의 확률')
plt.ylabel('손실 (-log p)')
plt.title('확률에 따른 CrossEntropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n관찰:")
print("- 정답 확률이 1에 가까울수록 손실이 0에 가까움")
print("- 정답 확률이 0에 가까울수록 손실이 무한대로 증가")


---
# 5. 핵심 요약
---

## 이번 단원에서 배운 내용

### 1. 확률의 기본
- **확률**: 불확실성을 0~1 사이의 숫자로 표현
- **조건부 확률**: $P(A|B) = \frac{P(A \cap B)}{P(B)}$
- **베이즈 정리**: 사전 확률 → 사후 확률 업데이트

### 2. 확률 분포
- **이산 분포**: 이항 분포, 다항 분포
- **연속 분포**: 정규 분포 $\mathcal{N}(\mu, \sigma^2)$
- **정규 분포의 특징**: 종 모양, 68-95-99.7 규칙

### 3. 최대우도 추정 (MLE)
- **아이디어**: 데이터를 가장 잘 설명하는 파라미터 찾기
- **로그 우도**: $\log L(\theta | \mathbf{x}) = \sum \log P(x_i | \theta)$
- **손실 함수**: $\text{Loss} = -\log L$ (Negative Log-Likelihood)

### 4. 손실 함수와 확률 분포의 연결
- **MSE ↔ 정규 분포**: 오차가 정규분포를 따른다고 가정
- **CrossEntropy ↔ 다항 분포**: 레이블이 다항분포를 따른다고 가정
- **핵심**: 손실 함수는 확률 가정에서 자연스럽게 유도됨

### 5. 실전 연결
- `nn.MSELoss()`: 회귀 문제 (정규 분포 가정)
- `nn.CrossEntropyLoss()`: 분류 문제 (다항 분포 가정)
- `F.softmax()`: 로짓을 확률 분포로 변환
- `BatchNorm`: 통계적 정규화 (평균 0, 분산 1)

## 다음 단원 미리보기

**다음 단원 (01)**: 텐서 기초

- 수학 기초를 다졌으니, 이제 PyTorch의 핵심 데이터 구조인 텐서를 배워봅시다
- 텐서는 벡터와 행렬의 일반화입니다
- 모든 연산의 기본 단위입니다

수학적 기초가 탄탄해졌으니, 이제 본격적으로 PyTorch를 다룰 준비가 되었습니다!
